# PlantHelixSeek Anno Gene-Structure Annotation (Arabidopsis showcase)

**Locus (illustrative):** `anno_locus=Chr1:5100001-5300000` — a 200 kb *Arabidopsis thaliana* Chr1 fragment (TAIR10).

**Methodology:** every scan, stitching, decode, match and metric rule in this notebook is transcribed
verbatim from the frozen Phase-6 locus-selection contract,
[`../plant_helixseek_shared/data/selection.md`](../plant_helixseek_shared/data/selection.md)
(the single source of truth for thresholds, tolerance bands and decode rules — nothing below is
re-derived or "improved").

**Frozen Anno contract summary (selection.md, "Verification contracts")**

- Model access through the dnallm public route only: `load_model_and_tokenizer(repo_id, TaskConfig, source="modelscope")`; the packaged registry (`dnallm/models/model_info.yaml`) is the single source of the repo id and the 17-BILOU label order. Forwards run under `torch.no_grad()`.
- Scan: 8192 bp windows / 4096 stride, BOTH strands, batch 1 (12.99 GB peak measured through the eager-attention dnallm route).
- Token alignment: `logits[:, 1:window+1, :]` (BOS offset; measured logits shape `(1, 8194, 17)`).
- Stitching (upstream middle-region rule): the first window contributes `[0, 6144)`; middle windows `[start+2048, start+6144)`; a tail window (start = len-8192) contributes `[start+2048, len)` and WINS where its core overlaps the previous core; full coverage is asserted. No probability averaging — predictions are stitched directly.
- Reverse strand: the reverse-complement window's labels are B<->L swapped via the upstream 17-element permutation (`predict_genome_multigpu.py:97-101`) and then reversed to plus-strand coordinates; plus- and minus-strand label tracks stay SEPARATE — a predicted segment's strand is the track it was decoded from.
- Decode (frozen argmax; the upstream viterbi+ORF path is deliberately NOT ported — the floors were calibrated on argmax): per-base argmax per strand; a predicted CDS segment is a MAXIMAL RUN of consecutive positions whose label is in {B-CDS, I-CDS, L-CDS, U-CDS}.
- Match rule: predicted segment <-> truth CDS row (Parent = mRNA), same strand, reciprocal overlap >= 0.5 (overlap/predicted_len AND overlap/truth_len); greedy in predicted-start order, first unmatched truth row wins.
- Metrics: pooled exon-level F1 across the locus; per-gene exon F1 for mRNA-contained genes; nucleotide-level sensitivity / precision / F1.

**Thresholds and tolerance bands (copied from selection.md):**

| Metric | Selection threshold | Tolerance band (assertion) | Observed (selection run) |
|---|---|---|---|
| Anno gene models with exon-F1 >= 0.8 | >= 3 genes | >= 3 genes | 59 |
| Negative Anno genic-base fraction (intergenic) | <= 0.1 | [0.00, 0.1] | 0.0000 |

**Environment:** the exact transformers / torch / flash-linear-attention versions this notebook ran
with are printed by the first code cell as `transformers_version=`, `torch_version=` and
`fla_version=` key=value lines.

**Disclaimer (illustrative loci):** all results in this notebook are computed on ONE illustrative
200 kb locus (plus one 20 kb intergenic negative-control window) selected by the Phase-6 methodology.
They demonstrate the workflow on an illustrative locus, not genome-wide accuracy — see
selection.md for the selection methodology and the meaning of each tolerance band.

In [1]:
# Hard environment guard (D-16): the PlantHelixSeek remote code silently falls
# back to a pure-PyTorch non-KDA path when flash-linear-attention is absent,
# producing positionally-dead outputs -- absence is a hard error, never a
# silent degradation. The spec probe is deliberate (never a bare module
# import): the fast-lane test tests/examples/test_examples.py execs every
# import statement in this notebook on legs without the fla extra.
import importlib.metadata
import importlib.util

import torch
import transformers

if importlib.util.find_spec("fla") is None:
    raise RuntimeError(
        "flash-linear-attention (fla) is not installed: the PlantHelixSeek "
        "kernels would silently fall back to the non-KDA path and produce "
        "positionally-dead outputs (see README section Flash-Linear-Attention). "
        "Install with: uv pip install -e '.[fla]'"
    )

# The fla version is read via importlib.metadata.version -- a call, not an
# import statement, so no import node ever names the fla module.
print(f"transformers_version={transformers.__version__}")
print(f"torch_version={torch.__version__}")
print(f"fla_version={importlib.metadata.version('flash-linear-attention')}")
print(f"cuda_device={torch.cuda.get_device_name(0)}")

transformers_version=5.17.0
torch_version=2.11.0+cu130
fla_version=0.5.2
cuda_device=NVIDIA GB10


## Load the checkpoint through the dnallm public route

The repo id and the 17-BILOU label order come from the packaged registry entry for
`zhangtaolab/PlantHelixSeek-Anno` (frozen in Phase 6 with the checkpoint commit sha recorded in
`dnallm/models/model_info.yaml`); the checkpoint is fetched from ModelScope through the generic
`load_model_and_tokenizer` dispatch (no special handler). The token task asks the model for
per-base logits: one 17-way distribution per sequence position.

In [2]:
from pathlib import Path

import dnallm
import yaml
from dnallm.configuration.configs import TaskConfig
from dnallm.models.model import load_model_and_tokenizer

REPO_ID = "zhangtaolab/PlantHelixSeek-Anno"

# Same registry-read shape as tests/models/test_plant_helixseek_smoke.py:
# the packaged yaml is the single source of repo id + label order.
registry_path = Path(dnallm.__file__).parent / "models" / "model_info.yaml"
registry = yaml.safe_load(registry_path.read_text(encoding="utf-8"))
entries = [e for e in registry["finetuned"] if e.get("model") == REPO_ID]
assert len(entries) == 1, f"expected exactly one registry entry for {REPO_ID}"
task = entries[0]["task"]
label_names = task["label_names"]
print(f"registry_task_type={task['task_type']}")
print(f"registry_num_labels={task['num_labels']}")
print(f"registry_label_names={label_names}")

task_config = TaskConfig(
    task_type=task["task_type"],
    num_labels=task["num_labels"],
    label_names=task["label_names"],
    threshold=task["threshold"],
)
model, tokenizer = load_model_and_tokenizer(REPO_ID, task_config, source="modelscope")
print(f"model_num_labels={model.config.num_labels}")
print(f"model_id2label={model.config.id2label}")
print(f"model_device={model.device}")

registry_task_type=token

registry_num_labels=17

registry_label_names=['O', 'B-CDS', 'I-CDS', 'L-CDS', 'U-CDS', 'B-INTRON', 'I-INTRON', 'L-INTRON', 'U-INTRON', 'B-UTR5', 'I-UTR5', 'L-UTR5', 'U-UTR5', 'B-UTR3', 'I-UTR3', 'L-UTR3', 'U-UTR3']

22:37:15 - dnallm.utils.support - 

INFO

 - Model files are stored in /home/forrest/.cache/modelscope/hub/models/zhangtaolab/PlantHelixSeek-Anno


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

model_num_labels=17

model_id2label={0: 'O', 1: 'B-CDS', 2: 'I-CDS', 3: 'L-CDS', 4: 'U-CDS', 5: 'B-INTRON', 6: 'I-INTRON', 7: 'L-INTRON', 8: 'U-INTRON', 9: 'B-UTR5', 10: 'I-UTR5', 11: 'L-UTR5', 12: 'U-UTR5', 13: 'B-UTR3', 14: 'I-UTR3', 15: 'L-UTR3', 16: 'U-UTR3'}

model_device=cuda:0

## Load the committed showcase data

The 200 kb locus FASTA carries its genomic interval in its header — the genomic offset is parsed
from that header (never hardcoded a second time), and it applies to PREDICTIONS only (the fragment
is the local-coordinate artifact; the truth rows are already genomic). The TAIR10 truth slice is
read DIRECTLY: the committed file already holds the fully-contained truth rows verbatim — never
re-sliced, never merged, never re-sorted. CDS rows whose FIRST `Parent=` value names an mRNA are
the truth segments (TAIR10 joins a `-Protein` partner after the mRNA id — `parse_gff_attributes`
splits the comma-joined value and the first entry wins); `mRNA` rows provide the per-gene spans.

In [3]:
import re

from dnallm.utils import fetch_sequence, gff1_to_half_open, parse_gff_attributes

fasta_path = Path("data/chr1_5100001_5300000.fas")
header = fasta_path.read_text(encoding="utf-8").splitlines()[0]
locus_match = re.search(r"\[(\d+),\s*(\d+)\]", header)
locus_start, locus_end = int(locus_match.group(1)), int(locus_match.group(2))
# Genomic offset (single source: the FASTA header interval, 1-based closed).
genomic_offset = locus_start - 1
locus_length = locus_end - locus_start + 1
print(f"anno_locus=Chr1:{locus_start}-{locus_end}")

# Fragment-local coordinates: the committed .fas holds exactly the locus.
sequence = fetch_sequence(str(fasta_path), "Chr1", 1, locus_length)
assert len(sequence) == locus_length

# Truth: the committed TAIR10 slice, verbatim rows, source order preserved.
truth_gff3 = Path("data/TAIR10_GFF3_chr1_5100001_5300000.gff3")
truth_rows = [
    row
    for row in truth_gff3.read_text(encoding="utf-8").splitlines()
    if row.strip() and not row.startswith("#")
]
truth_cds = []  # (start0, end0, strand, mrna_id) -- genomic half-open
gene_spans = []  # (start1, end1, strand, mrna_id) -- genomic 1-based closed
for row in truth_rows:
    cols = row.split("\t")
    attrs = parse_gff_attributes(cols[8])
    if cols[2] == "CDS":
        start0, end0 = gff1_to_half_open(int(cols[3]), int(cols[4]))
        truth_cds.append((start0, end0, cols[6], attrs.get("Parent", [""])[0]))
    elif cols[2] == "mRNA":
        gene_spans.append((int(cols[3]), int(cols[4]), cols[6], attrs.get("ID", [""])[0]))
print(f"truth_rows_parsed={len(truth_rows)}")
print(f"truth_cds_rows={len(truth_cds)}")
print(f"truth_mrna_rows={len(gene_spans)}")

anno_locus=Chr1:5100001-5300000

truth_rows_parsed=1522

truth_cds_rows=526

truth_mrna_rows=91

## Window plan — the frozen stitching rule

8192 bp windows advance by a 4096 bp stride; each window only "owns" its central core (the 2048 bp
margin is dropped on each side — the upstream middle-region rule). The FIRST window contributes
`[0, 6144)`; middle windows contribute `[start+2048, start+6144)`; when the locus end is not on the
4096 grid, a TAIL window anchored at `len-8192` contributes `[start+2048, len)` and WINS where its
core overlaps the previous core (it is processed last). Predictions are stitched directly — never
probability averaging. Full coverage of the locus is asserted (a sanctioned structural assert, not
a metric floor).

In [4]:
import time

import numpy as np

# Frozen scan contract (selection.md): 8192 bp / 4096 stride, batch 1.
# Batch 1 is the eager-attention ceiling through the dnallm route (dnallm
# forces attn_implementation="eager"; measured 12.99 GB peak per forward).
window, stride, batch_size = 8192, 4096, 1
margin = (window - stride) // 2  # 2048 dropped per side per window


def plan_windows(n):
    """Stride windows + optional tail window; returns (starts, cores).

    The first window contributes [0, 6144); middle windows [s+2048, s+6144);
    a tail window (start = n - window) contributes [start+2048, n). The tail
    is processed LAST by callers, so where its core overlaps the previous
    core the tail window wins (frozen selection.md stitching rule).
    """
    starts = list(range(0, n - window + 1, stride))
    cores = []
    for i, s in enumerate(starts):
        c0 = 0 if i == 0 else s + margin
        cores.append((c0, s + window - margin))
    if starts and starts[-1] + window < n:
        tail_start = n - window
        starts.append(tail_start)
        cores.append((tail_start + margin, n))
    return starts, cores


scan_starts, scan_cores = plan_windows(locus_length)

# Structural assert (sanctioned): the stitched cores cover the whole locus.
core_covered = np.zeros(locus_length, dtype=bool)
for c0, c1 in scan_cores:
    core_covered[c0:c1] = True
assert core_covered.all(), "stitching left uncovered bases (SHOW-02 non-emptiness)"
print(f"scan_windows={len(scan_starts)}")
print(f"stitch_cores={len(scan_cores)}")
print(f"tail_window_start={scan_starts[-1]}")
print(f"tail_window_core=({scan_cores[-1][0]}, {scan_cores[-1][1]})")

scan_windows=48

stitch_cores=48

tail_window_start=191808

tail_window_core=(193856, 200000)

## Plus-strand scan (batch 1, under `torch.no_grad()`)

Each window is tokenized and forwarded alone (batch 1) under `torch.no_grad()` — mandatory: an
autograd graph over the 8192-token eager-attention window OOMs the box. The checkpoint emits one
logit row per input token PLUS a leading BOS and trailing EOS position (measured shape
`(1, 8194, 17)`), so base `i` of the window reads `logits[0, 1 + i, :]` — the BOS-offset token
alignment. Per-base argmax labels are written directly into each window's core slice (stitching
without averaging).

In [5]:
def predict_window_labels(sub):
    """Forward ONE window at batch 1 under no_grad; return per-base argmax labels.

    BOS offset: logits carry a leading BOS position (and a trailing EOS), so
    base i of the window is logits[0, 1 + i, :] -- selection.md token
    alignment, upstream predict_genome_multigpu.py:308.
    """
    enc = tokenizer(sub, return_tensors="pt")
    with torch.no_grad():
        logits = model(
            input_ids=enc["input_ids"].to(model.device),
            attention_mask=enc["attention_mask"].to(model.device),
        ).logits
    return logits[0, 1 : window + 1, :].argmax(dim=-1).cpu().numpy().astype(np.int64)


plus_started = time.perf_counter()
plus_labels = np.zeros(locus_length, dtype=np.int64)
plus_written = np.zeros(locus_length, dtype=bool)
for s, (c0, c1) in zip(scan_starts, scan_cores):
    labels = predict_window_labels(sequence[s : s + window])
    plus_labels[c0:c1] = labels[c0 - s : c1 - s]  # stitch directly into the core
    plus_written[c0:c1] = True
assert plus_written.all(), "plus-strand stitching left uncovered bases"
print(f"plus_strand_windows={len(scan_starts)}")
print(f"plus_strand_scanned_bases={int(plus_written.sum())}")
print(f"plus_strand_seconds={time.perf_counter() - plus_started:.1f}")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


HelixSeek requires an initialized cache to return a cache. None was provided, so no cache will be returned.


plus_strand_windows=48

plus_strand_scanned_bases=200000

plus_strand_seconds=293.6

## Minus-strand scan (reverse complement + the upstream B<->L permutation)

The same windows are reverse-complemented (`dnallm.utils.sequence.reverse_complement`) and
forwarded again: the model now reads the minus strand in its own 5'→3' direction, so its B/I/L
boundaries are oriented along the minus strand. Mapping back to plus-strand coordinates takes the
upstream 17-element permutation — each entity's B and L swap, I/O/U are fixed points — followed by
reversing the label vector. Plus- and minus-strand tracks stay SEPARATE: a predicted segment's
strand is the track it was decoded from, never a merge of the two.

In [6]:
from dnallm.utils.sequence import reverse_complement

# Upstream 17-element B<->L permutation, transcribed verbatim from
# PlantHelixSeek predict_genome_multigpu.py:97-101 (selection.md:43; the
# frozen label order is O, {B,I,L,U}-CDS, {B,I,L,U}-INTRON, {B,I,L,U}-UTR5,
# {B,I,L,U}-UTR3). Index = model label on the reverse-complement window,
# value = the same entity state in plus-strand reading orientation.
_B_SWAP_L = [0, 3, 2, 1, 4, 7, 6, 5, 8, 11, 10, 9, 12, 15, 14, 13, 16]
# _B_SWAP_L (and the intergenic control's labels != 0 = O reading) is
# POSITIONAL: pin the registry label order the permutation was transcribed
# against (selection.md:43) so a registry reorder fails here instead of
# silently mis-mapping bases while label_names.index() follows the new order.
assert label_names == [
    "O", "B-CDS", "I-CDS", "L-CDS", "U-CDS",
    "B-INTRON", "I-INTRON", "L-INTRON", "U-INTRON",
    "B-UTR5", "I-UTR5", "L-UTR5", "U-UTR5",
    "B-UTR3", "I-UTR3", "L-UTR3", "U-UTR3",
], "registry label order drifted from the frozen _B_SWAP_L permutation"
swap = np.array(_B_SWAP_L, dtype=np.int64)

minus_started = time.perf_counter()
minus_labels = np.zeros(locus_length, dtype=np.int64)
minus_written = np.zeros(locus_length, dtype=bool)
for s, (c0, c1) in zip(scan_starts, scan_cores):
    sub = reverse_complement(sequence[s : s + window])  # both flags default True
    labels = predict_window_labels(sub)
    labels = swap[labels][::-1]  # B<->L swap, then reverse to plus-strand coordinates
    minus_labels[c0:c1] = labels[c0 - s : c1 - s]
    minus_written[c0:c1] = True
assert minus_written.all(), "minus-strand stitching left uncovered bases"

# The two strand tracks stay SEPARATE (frozen contract).
label_tracks = {"+": plus_labels, "-": minus_labels}
print(f"minus_strand_windows={len(scan_starts)}")
print(f"minus_strand_scanned_bases={int(minus_written.sum())}")
print(f"minus_strand_seconds={time.perf_counter() - minus_started:.1f}")

minus_strand_windows=48

minus_strand_scanned_bases=200000

minus_strand_seconds=292.9

## Decode predicted CDS segments (frozen argmax BILOU rule)

The per-base argmax is already computed per strand; a predicted CDS segment is a MAXIMAL RUN of
consecutive positions whose label is in the CDS family {B-CDS, I-CDS, L-CDS, U-CDS}. Segments are
locus-local 0-based half-open intervals; a segment's strand is the track it was decoded from.
(The upstream viterbi+ORF decoder is deliberately NOT ported — the floors were calibrated on this
argmax rule; any other decode invalidates every band.)

In [7]:
CDS_FAMILY = ("B-CDS", "I-CDS", "L-CDS", "U-CDS")
cds_label_idx = {label_names.index(name) for name in CDS_FAMILY}
print(f"cds_label_indices={sorted(cds_label_idx)}")


def decode_cds_segments(labels):
    """Maximal runs of CDS-family labels -> locus-local 0-based half-open intervals."""
    segments = []
    n = len(labels)
    i = 0
    while i < n:
        if int(labels[i]) in cds_label_idx:
            j = i
            while j < n and int(labels[j]) in cds_label_idx:
                j += 1
            segments.append((i, j))
            i = j
        else:
            i += 1
    return segments


predicted_segments = {strand: decode_cds_segments(label_tracks[strand]) for strand in ("+", "-")}
print(f"pred_segments_plus={len(predicted_segments['+'])}")
print(f"pred_segments_minus={len(predicted_segments['-'])}")
print(f"pred_segments_total={len(predicted_segments['+']) + len(predicted_segments['-'])}")

cds_label_indices=[1, 2, 3, 4]

pred_segments_plus=131

pred_segments_minus=263

pred_segments_total=394

## GFF3 emission and structural validation

Predicted segments are lifted from locus-local to genomic coordinates with the FASTA-header offset
(the +5.1 Mb shift applies to predictions only — truth rows are already genomic) and converted to
1-based closed coordinates via `half_open_to_gff1`, then written as a 9-column GFF3
(`seqid source type start end score strand phase attributes`) under `outputs/` (gitignored). The
file is then re-parsed and validated STRUCTURALLY in-notebook: 9 tab-separated columns, integer
coordinates inside the locus, strand within the {+, -, .} set — a sanctioned structural assert, not
a metric floor. `Parent=` names a per-segment placeholder model: the frozen argmax decode does not
phase or group segments into genes.

In [8]:
from dnallm.utils import half_open_to_gff1

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)
gff3_path = outputs_dir / "predicted_cds_segments.gff3"

gff3_rows = []
segment_index = 0
for strand in ("+", "-"):
    for local_start, local_end in predicted_segments[strand]:
        segment_index += 1
        g1_start, g1_end = half_open_to_gff1(
            local_start + genomic_offset, local_end + genomic_offset
        )
        gff3_rows.append(
            f"Chr1\tPlantHelixSeek-Anno\tCDS\t{g1_start}\t{g1_end}\t.\t{strand}\t."
            f"\tID=predicted_cds_{segment_index};Parent=predicted_gene_{segment_index}"
        )
gff3_path.write_text("\n".join(gff3_rows) + "\n", encoding="utf-8")

# Structural validation (sanctioned assert): re-parse the emitted file.
validated_rows = 0
for row in gff3_path.read_text(encoding="utf-8").splitlines():
    if not row.strip():
        continue
    cols = row.split("\t")
    assert len(cols) == 9, f"emitted GFF3 row does not have 9 columns: {row!r}"
    g1_start, g1_end = int(cols[3]), int(cols[4])  # raises on non-integer coords
    assert locus_start <= g1_start <= g1_end <= locus_end, (
        f"emitted coordinates outside the locus: {g1_start}-{g1_end}"
    )
    assert cols[6] in {"+", "-", "."}, f"invalid strand: {cols[6]!r}"
    assert cols[8], "empty attributes column"
    validated_rows += 1
print(f"pred_gff3_rows={validated_rows}")

pred_gff3_rows=394

## Exon-level and nucleotide-level metrics vs the TAIR10 truth

Frozen match rule: each predicted segment is matched to a truth CDS row on the SAME strand with
reciprocal overlap >= 0.5 (overlap/predicted_len AND overlap/truth_len both >= 0.5), greedily in
predicted-start order with the first unmatched truth row winning. Pooled exon-level F1 counts
TP = matched truth rows, FP = unmatched predicted segments, FN = unmatched truth rows. Per-gene
exon F1 scores every mRNA carrying CDS rows in the committed slice (TP/FN from that gene's truth
rows; FP = that gene's unmatched predicted segments on the gene's strand whose midpoint lies inside
the gene span); the selection floor counts genes with per-gene exon-F1 >= 0.8. The nucleotide-level
sensitivity / precision / F1 compare per-strand CDS base sets (pooled TP/FP/FN across strands) —
an evidence metric, not part of the selection bands. Truth rows are read exactly as committed:
never merged, never re-sorted.

In [9]:
MATCH_RECIP = 0.5  # frozen reciprocal-overlap match rule
EXON_F1_FLOOR = 0.8  # per-gene floor for the genes_above_floor count


def to_genomic(intervals):
    """Locus-local 0-based half-open -> genomic half-open (offset addition)."""
    return [(s + genomic_offset, e + genomic_offset) for s, e in intervals]


def greedy_match(preds, truths):
    """Greedy reciprocal-overlap >= 0.5 matching.

    Iteration is predicted-start ascending (first unmatched truth row wins);
    returns (pred_index, truth_index) pairs.
    """
    matches = []
    used = set()
    for pi, (ps, pe) in sorted(enumerate(preds), key=lambda item: item[1]):
        for ti, (ts, te) in enumerate(truths):
            if ti in used:
                continue
            overlap = min(pe, te) - max(ps, ts)
            if overlap <= 0:
                continue
            if overlap / (pe - ps) >= MATCH_RECIP and overlap / (te - ts) >= MATCH_RECIP:
                matches.append((pi, ti))
                used.add(ti)
                break
    return matches


matched_truth = set()
matched_pred = set()
for strand in ("+", "-"):
    truth_idx = [i for i, t in enumerate(truth_cds) if t[2] == strand]
    pred_genomic = to_genomic(predicted_segments[strand])
    for pi, ti in greedy_match(pred_genomic, [truth_cds[i][:2] for i in truth_idx]):
        matched_pred.add((strand, pi))
        matched_truth.add(truth_idx[ti])

tp = len(matched_truth)
fp = len(predicted_segments["+"]) + len(predicted_segments["-"]) - tp
fn = len(truth_cds) - tp
pooled_denom = 2 * tp + fp + fn
exon_f1_value = (2 * tp / pooled_denom) if pooled_denom else 1.0

# Per-gene exon F1 (frozen rule): gene = mRNA with CDS rows in the slice;
# FP scoped to the gene's strand and span (midpoint rule, half-open span).
truth_by_gene = {}
for i, t in enumerate(truth_cds):
    truth_by_gene.setdefault(t[3], []).append(i)
gene_f1_scores = []
for g1_start, g1_end, g_strand, g_id in gene_spans:
    rows = truth_by_gene.get(g_id)
    if not rows:
        continue
    tp_g = sum(1 for i in rows if i in matched_truth)
    fn_g = len(rows) - tp_g
    gs0, ge0 = gff1_to_half_open(g1_start, g1_end)
    fp_g = 0
    for pi, interval in enumerate(to_genomic(predicted_segments[g_strand])):
        if (g_strand, pi) in matched_pred:
            continue
        if gs0 <= (interval[0] + interval[1]) / 2 < ge0:
            fp_g += 1
    d_g = 2 * tp_g + fp_g + fn_g
    gene_f1_scores.append((g_id, (2 * tp_g / d_g) if d_g else 1.0))
genes_above_floor_value = sum(1 for _, f1 in gene_f1_scores if f1 >= EXON_F1_FLOOR)

# Nucleotide level (evidence-only): per-strand CDS base masks, pooled counts.
nt_tp = nt_fp = nt_fn = 0
for strand in ("+", "-"):
    pred_mask = np.zeros(locus_length, dtype=bool)
    for s, e in predicted_segments[strand]:
        pred_mask[s:e] = True
    truth_mask = np.zeros(locus_length, dtype=bool)
    for t_start0, t_end0, t_strand, _ in truth_cds:
        if t_strand == strand:
            truth_mask[t_start0 - genomic_offset : t_end0 - genomic_offset] = True
    nt_tp += int((pred_mask & truth_mask).sum())
    nt_fp += int((pred_mask & ~truth_mask).sum())
    nt_fn += int((~pred_mask & truth_mask).sum())
nt_sensitivity = nt_tp / (nt_tp + nt_fn) if (nt_tp + nt_fn) else 1.0
nt_precision = nt_tp / (nt_tp + nt_fp) if (nt_tp + nt_fp) else 1.0
nt_f1 = 2 * nt_tp / (2 * nt_tp + nt_fp + nt_fn) if (2 * nt_tp + nt_fp + nt_fn) else 1.0

print(f"exon_f1={exon_f1_value:.4f}")
print(f"genes_above_floor={genes_above_floor_value}")
print(f"n_truth_cds={len(truth_cds)}")
print(f"n_pred_segments={len(predicted_segments['+']) + len(predicted_segments['-'])}")
print(f"tp={tp}")
print(f"fp={fp}")
print(f"fn={fn}")
print(f"n_genes={len(gene_f1_scores)}")
print(f"nt_sensitivity={nt_sensitivity:.4f}")
print(f"nt_precision={nt_precision:.4f}")
print(f"nt_f1={nt_f1:.4f}")

exon_f1=0.7522

genes_above_floor=59

n_truth_cds=526

n_pred_segments=394

tp=346

...

fn=180

n_genes=91

nt_sensitivity=0.9802

nt_precision=0.9412

nt_f1=0.9603

## Gene-model diagrams (predicted vs truth)

One lane per truth mRNA (orange CDS blocks on a grey intron backbone), strand-separated panels,
with the predicted CDS segments (blue) as strand tracks on top — all on the shared locus
coordinate axis. The embedded vega output renders in the GitHub notebook viewer.

In [10]:
import json

import altair as alt
import pandas as pd
import vl_convert as vlc
from IPython.display import display

alt.data_transformers.enable("default", max_rows=None)

# Genomic 1-based closed display coordinates throughout.
truth_cds_df = pd.DataFrame(
    [
        {"gene": mrna_id, "start": s0 + 1, "end": e0}
        for s0, e0, strand, mrna_id in truth_cds
        if strand == "+"
    ]
    + [
        {"gene": mrna_id, "start": s0 + 1, "end": e0}
        for s0, e0, strand, mrna_id in truth_cds
        if strand == "-"
    ]
)
span_records = {"+": [], "-": []}
for g1_start, g1_end, g_strand, g_id in gene_spans:
    span_records[g_strand].append({"gene": g_id, "start": g1_start, "end": g1_end})
pred_records = []
for strand in ("+", "-"):
    for s, e in predicted_segments[strand]:
        pred_records.append({"strand": strand, "start": s + genomic_offset + 1, "end": e + genomic_offset})
pred_track_df = pd.DataFrame(pred_records)

gene_orders = {}
for strand in ("+", "-"):
    ordered = sorted(span_records[strand], key=lambda r: r["start"])
    gene_orders[strand] = [r["gene"] for r in ordered]

predicted_track = (
    alt.Chart(pred_track_df)
    .mark_rect(color="#4c78a8")
    .encode(
        x=alt.X("start:Q").title(None).axis(format="s", labels=False),
        x2=alt.X2("end:Q"),
        y=alt.Y("strand:N").title("predicted CDS segments"),
    )
    .properties(height=60)
)


def gene_panel(strand):
    spans_df = pd.DataFrame(span_records[strand])
    order = gene_orders[strand]
    cds_df = truth_cds_df[truth_cds_df["gene"].isin(order)]
    backbone = (
        alt.Chart(spans_df)
        .mark_rule(color="#999999")
        .encode(
            x=alt.X("start:Q").title(None).axis(format="s", labels=False),
            x2=alt.X2("end:Q"),
            y=alt.Y("gene:N").sort(order).axis(None),
        )
    )
    blocks = (
        alt.Chart(cds_df)
        .mark_rect(color="#f58518")
        .encode(
            x=alt.X("start:Q").title(None).axis(format="s", labels=False),
            x2=alt.X2("end:Q"),
            y=alt.Y("gene:N").sort(order).axis(None),
        )
    )
    return (
        (backbone + blocks)
        .properties(
            height={"step": 7},
            title=f"truth gene models -- {strand} strand ({len(order)} genes)",
        )
        .interactive()
    )


x_axis = (
    alt.Chart(pd.DataFrame({"x": [locus_start, locus_end]}))
    .mark_tick()
    .encode(x=alt.X("x:Q").title("Chr1 position (bp)").axis(format="s"))
    .properties(height=10)
)

gene_model_chart = alt.vconcat(
    predicted_track,
    gene_panel("+"),
    gene_panel("-"),
    x_axis,
).resolve_scale(x="shared")

# Embed the figure as inline vega JSON (no external files, no binaries): the
# compiled vega spec renders in the GitHub notebook viewer and the native
# vega-lite spec rides along for newer front-ends. Rounding coordinates and
# the per-gene-lane layout keep the embedded payloads inside the 2 MB budget.
vegalite_spec = json.loads(gene_model_chart.to_json())
vega_spec = vlc.vegalite_to_vega(vegalite_spec)
display(
    {
        "application/vnd.vega.v6+json": vega_spec,
        "application/vnd.vegalite.v6.json": json.dumps(vegalite_spec),
        "text/plain": (
            "PlantHelixSeek Anno predicted CDS segments (blue) vs TAIR10 truth "
            "gene models (orange CDS blocks on grey intron backbones)"
        ),
    },
    raw=True,
)
print(f"figure_truth_genes_plus={len(gene_orders['+'])}")
print(f"figure_truth_genes_minus={len(gene_orders['-'])}")

/home/forrest/Github/DNALLM/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3823: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


PlantHelixSeek Anno predicted CDS segments (blue) vs TAIR10 truth gene models (orange CDS blocks on grey intron backbones)

figure_truth_genes_plus=35

figure_truth_genes_minus=56

Predicted CDS segments (top, blue) track the TAIR10 exon structure closely on this locus:
the pooled exon-level F1 and the per-gene scores above quantify what is visible here — most truth
gene lanes have a matching predicted block structure, with the mismatch concentrated in short or
low-coverage genes. Illustrative locus, not genome-wide accuracy.

## Negative control: intergenic window

Phase 6 selected a 20 kb window inside the largest fully feature-free TAIR10 gap in the scan
region (`Chr1:14953292-14973291`) as the expected near-zero-signal control. The assertion metric
is the predicted genic (non-O, either strand) base fraction under the SAME frozen 8192/4096
both-strand scan + stitching rule. The selection run recorded `neg_anno_fraction=0.0000`
(selection.md tolerance band [0.00, 0.1]). The committed intergenic TAIR10 gene slice
[`../plant_helixseek_shared/data/TAIR10_GFF3_chr1_14953292_14973291.gff3`](../plant_helixseek_shared/data/TAIR10_GFF3_chr1_14953292_14973291.gff3)
is a ZERO-ROW file — rendered as zero, never dropped — and is read below as the rendered-as-zero
evidence that the window is feature-free.

In [11]:
intergenic_fasta = Path("../plant_helixseek_shared/data/chr1_14953292_14973291.fas")
intergenic_header = intergenic_fasta.read_text(encoding="utf-8").splitlines()[0]
intergenic_match = re.search(r"\[(\d+),\s*(\d+)\]", intergenic_header)
intergenic_start = int(intergenic_match.group(1))
intergenic_end = int(intergenic_match.group(2))
intergenic_length = intergenic_end - intergenic_start + 1
intergenic_sequence = fetch_sequence(str(intergenic_fasta), "Chr1", 1, intergenic_length)
assert len(intergenic_sequence) == intergenic_length
print(f"intergenic_locus=Chr1:{intergenic_start}-{intergenic_end}")

# Rendered-as-zero evidence: the committed intergenic truth slice holds no rows.
intergenic_truth_rows = [
    row
    for row in Path(
        "../plant_helixseek_shared/data/TAIR10_GFF3_chr1_14953292_14973291.gff3"
    )
    .read_text(encoding="utf-8")
    .splitlines()
    if row.strip() and not row.startswith("#")
]
print(f"intergenic_truth_rows={len(intergenic_truth_rows)}")

# Same frozen 8192/4096 both-strand scan + stitching on the intergenic window
# (reusing plan_windows / predict_window_labels / reverse_complement / swap).
neg_started = time.perf_counter()
intergenic_starts, intergenic_cores = plan_windows(intergenic_length)
intergenic_tracks = {}
for strand in ("+", "-"):
    labels = np.zeros(intergenic_length, dtype=np.int64)
    written = np.zeros(intergenic_length, dtype=bool)
    for s, (c0, c1) in zip(intergenic_starts, intergenic_cores):
        sub = intergenic_sequence[s : s + window]
        if strand == "-":
            sub = reverse_complement(sub)
        lab = predict_window_labels(sub)
        if strand == "-":
            lab = swap[lab][::-1]
        labels[c0:c1] = lab[c0 - s : c1 - s]
        written[c0:c1] = True
    assert written.all(), "intergenic stitching left uncovered bases"
    intergenic_tracks[strand] = labels

# Assertion metric (frozen): predicted genic (non-O, either strand) base fraction.
genic_mask = (intergenic_tracks["+"] != 0) | (intergenic_tracks["-"] != 0)
neg_anno_fraction_value = float(genic_mask.mean())
print(f"neg_anno_fraction={neg_anno_fraction_value:.4f}")
print(f"intergenic_scan_windows={len(intergenic_starts) * 2}")
print(f"intergenic_seconds={time.perf_counter() - neg_started:.1f}")

intergenic_locus=Chr1:14953292-14973291

intergenic_truth_rows=0

neg_anno_fraction=0.0000

intergenic_scan_windows=8

intergenic_seconds=48.8

## Observed vs the frozen contract

`selection.md` is parsed at runtime (line-start key anchors). This notebook PRINTS the comparison
and asserts nothing on metric floors — the nightly test layer owns every assertion.

In [12]:
selection_text = Path("../plant_helixseek_shared/data/selection.md").read_text(
    encoding="utf-8"
)
frozen = {}
for key in ("exon_f1", "genes_above_floor", "neg_anno_fraction"):
    frozen_match = re.search(rf"^{key}=([0-9.]+)", selection_text, re.MULTILINE)
    if frozen_match is None:
        raise RuntimeError(f"selection.md is missing frozen key '{key}=' (parse guard)")
    frozen[key] = float(frozen_match.group(1))

genes_band = None
neg_band = None
for line in selection_text.splitlines():
    if line.startswith("| Anno gene models with exon-F1 >= 0.8 |") and genes_band is None:
        genes_match = re.search(r">=\s*(\d+)\s*genes", line)
        if genes_match is not None:
            genes_band = int(genes_match.group(1))
    if (
        line.startswith("| Negative Anno genic-base fraction (intergenic) |")
        and neg_band is None
    ):
        neg_match = re.search(r"\[([0-9.]+),\s*([0-9.]+)\]", line)
        if neg_match is not None:
            neg_band = (float(neg_match.group(1)), float(neg_match.group(2)))
if genes_band is None or neg_band is None:
    raise RuntimeError("selection.md band table parse guard failed for the Anno rows")

print(f"{'metric':<46}{'observed':>10}{'selection':>11}{'band':>14}")
print(
    f"{'exon_f1 (evidence only, no band)':<46}{exon_f1_value:>10.4f}"
    f"{frozen['exon_f1']:>11.4f}{'-':>14}"
)
genes_inside = genes_above_floor_value >= genes_band
print(
    f"{'Anno gene models with exon-F1 >= 0.8':<46}{genes_above_floor_value:>10}"
    f"{frozen['genes_above_floor']:>11.0f}{'>= ' + str(genes_band):>14}"
    f"{'inside' if genes_inside else 'OUTSIDE'}"
)
neg_inside = neg_band[0] <= neg_anno_fraction_value <= neg_band[1]
print(
    f"{'Negative Anno genic-base fraction (intergenic)':<46}{neg_anno_fraction_value:>10.4f}"
    f"{frozen['neg_anno_fraction']:>11.4f}"
    f"{'[' + format(neg_band[0], '.2f') + ', ' + format(neg_band[1], '.2f') + ']':>14}"
    f"{'inside' if neg_inside else 'OUTSIDE'}"
)

metric                                          observed  selection          band

exon_f1 (evidence only, no band)                  0.7522     0.7522             -

Anno gene models with exon-F1 >= 0.8                  59         59          >= 3inside

Negative Anno genic-base fraction (intergenic)    0.0000     0.0000  [0.00, 0.10]inside

## Summary

On the illustrative locus Chr1:5100001-5300000 the frozen-contract both-strand scan + stitching +
argmax BILOU decode + GFF3 emission reproduces the Phase-6 selection behavior (`exon_f1=`,
`genes_above_floor=`, `neg_anno_fraction=` stream lines above), and every rule used here is the
frozen Phase-6 selection contract — see
[`../plant_helixseek_shared/data/selection.md`](../plant_helixseek_shared/data/selection.md).
These results are computed on one illustrative locus, not genome-wide accuracy; the nightly
execution tests assert the tolerance bands parsed from that same document.